# QUERY 2
Nombre de banco, cuenta de origen y monto de la max. transacción USD de cada banco.

In [ ]:
# importamos dataset
import pandas as pd

CANTIDAD_FILAS = 100
RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCIONES = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO_CUENTAS = "LI-Small_accounts.csv"
RUTA_TRANSACCIONES_SAMPLE = f"{RUTA_DE_DATASETS}/transacciones_sample_pequeño.csv"
RUTA_CUENTAS_SAMPLE = f"{RUTA_DE_DATASETS}/accounts_sample_pequeño.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_CUENTAS}",
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)
columnas_cuentas_originales = cuentas_completas.columns.tolist()

def normalizar_bank_id(serie):
    return serie.str.strip().str.lstrip("0").replace("", "0")

transacciones = transacciones_completas.sample(n=CANTIDAD_FILAS, random_state=42)
guardar_csv = lambda df, nombre_archivo: df.to_csv(nombre_archivo, index=False)
guardar_csv(transacciones, RUTA_TRANSACCIONES_SAMPLE)

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)

transacciones_sample["From Bank Normalized"] = normalizar_bank_id(transacciones_sample["From Bank"])
transacciones_sample["To Bank Normalized"] = normalizar_bank_id(transacciones_sample["To Bank"])
cuentas_completas["Bank ID Normalized"] = normalizar_bank_id(cuentas_completas["Bank ID"])

cuentas_relevantes = pd.concat([
    transacciones_sample[["From Bank Normalized", "Account"]].rename(
        columns={"From Bank Normalized": "Bank ID Normalized", "Account": "Account Number"}
    ),
    transacciones_sample[["To Bank Normalized", "Account.1"]].rename(
        columns={"To Bank Normalized": "Bank ID Normalized", "Account.1": "Account Number"}
    )
], ignore_index=True).dropna().drop_duplicates()

cuentas_sample = cuentas_completas.merge(
    cuentas_relevantes,
    on=["Bank ID Normalized", "Account Number"],
    how="inner"
)[columnas_cuentas_originales]
guardar_csv(cuentas_sample, RUTA_CUENTAS_SAMPLE)

cuentas_sample.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #9,110080,804EF3640,800D8BC10,Corporation #40990
1,Dawn Bank,25526,814E1ED90,80033DFA0,Sole Proprietorship #72123
2,Savings Bank of Cleveland,3500,80616EE50,800297DA0,Sole Proprietorship #9143
3,Desert Bancorp,22031,8071B3FF0,8001C14E0,Corporation #10570
4,First Bank of Phoenix,17256,81C11F940,8004F6600,Sole Proprietorship #27071


In [21]:
RUTA_RESULTADO_QUERY2 = f"query2_result.csv"

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_sample = pd.read_csv(
    RUTA_CUENTAS_SAMPLE,
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)

transacciones_usd = transacciones_sample[
    transacciones_sample["Payment Currency"] == "US Dollar"
].copy()
transacciones_usd["From Bank Normalized"] = normalizar_bank_id(transacciones_usd["From Bank"])
transacciones_usd["Amount Paid"] = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid"])

# Un resultado por Bank ID (único): máximo de Amount Paid y la cuenta que lo generó
bancos = cuentas_sample[["Bank ID", "Bank Name"]].copy()
bancos["From Bank Normalized"] = normalizar_bank_id(bancos["Bank ID"])
bancos = bancos[["From Bank Normalized", "Bank ID", "Bank Name"]].drop_duplicates(subset=["From Bank Normalized"])

transacciones_con_banco = transacciones_usd.merge(bancos, on="From Bank Normalized", how="inner")

idx_maximos = transacciones_con_banco.groupby("From Bank Normalized")["Amount Paid"].idxmax()
resultado_query2 = transacciones_con_banco.loc[idx_maximos, ["Bank ID", "Bank Name", "Account", "Amount Paid"]].rename(columns={
    "Amount Paid": "Max Amount"
}).sort_values(by="Bank ID").reset_index(drop=True)

guardar_csv(resultado_query2, RUTA_RESULTADO_QUERY2)
resultado_query2.head()

,Bank ID,Bank Name,Account,Max Amount
0,1,First Bank of Portland,8027C2CE0,79.12
1,11046,Bank of Indianapolis,800652AF0,498.38
2,11047,National Bank of Philadelphia,806204650,4275.74
3,1110,Capital Trust Bank,800C77D10,293.30
4,1120,First Bank of the South,80102E920,164.66
